In [222]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

torch.manual_seed(42)

In [223]:
url = "http://data.gdeltproject.org/events/20220101.export.CSV.zip"
df = pd.read_csv(url, sep='\t', header=None, compression='zip', nrows=20000)

# Select useful columns
df = df[[1, 5, 7, 26]]  # time, src, dst, relation
df.columns = ['time', 'src', 'dst', 'rel']
rel_types = df['rel'].unique()
rel2id = {r: i for i, r in enumerate(rel_types)}

df['rel'] = df['rel'].map(rel2id)
num_rels = len(rel2id)
df = df.dropna()
df['time'] = pd.to_datetime(df['time'], format='%Y%m%d')

print(df.head())

        time     src  dst  rel
2 2021-01-01  GBRCVL  GBR    1
3 2021-01-01     USA  USA    0
6 2021-12-25     NIC  NIC    4
7 2021-12-25     NIC  NIC    4
8 2021-12-25  NICGOV  NIC    4


In [224]:
nodes = pd.concat([df['src'], df['dst']]).unique()
node2id = {n: i for i, n in enumerate(nodes)}

df['src'] = df['src'].map(node2id)
df['dst'] = df['dst'].map(node2id)

num_nodes = len(node2id)
print("Num nodes:", num_nodes)

Num nodes: 515


In [225]:
def smart_negative_sampling(df, num_nodes):
    active_nodes = df['dst'].values
    neg_dst = np.random.choice(active_nodes, size=len(df))
    return df['src'].values, neg_dst

In [226]:
class RelationalTGN(nn.Module):
    def __init__(self, num_nodes, num_rels, emb_dim):
        super().__init__()

        self.memory = nn.Embedding(num_nodes, emb_dim)
        self.rel_emb = nn.Embedding(num_rels, emb_dim)

        self.gru = nn.GRUCell(emb_dim, emb_dim)

        self.msg_mlp = nn.Sequential(
            nn.Linear(emb_dim * 3 + 6, emb_dim),  # src + dst + rel + time(2)
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim)
        )

        self.scorer = nn.Sequential(
            nn.Linear(emb_dim * 2 + 6, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, 1)
        )

        nn.init.xavier_uniform_(self.memory.weight)
        nn.init.xavier_uniform_(self.rel_emb.weight)

    def time_encoding(self, t):
        freqs = torch.tensor([1.0, 10.0, 100.0], device=t.device)

        t = t.unsqueeze(1)              # [B, 1]
        freqs = freqs.unsqueeze(0)      # [1, 3]

        angles = t * freqs              # [B, 3]

        enc = torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)  # [B, 6]
        return enc

    def compute_message(self, src_mem, dst_mem, rel_emb, t):
        t_enc = self.time_encoding(t)
        msg_input = torch.cat([src_mem, dst_mem, rel_emb, t_enc], dim=1)
        return self.msg_mlp(msg_input)

    def update_memory(self, node_ids, messages):
        old_mem = self.memory(node_ids)
        new_mem = self.gru(messages, old_mem)
        self.memory.weight.data[node_ids] = new_mem

    def forward(self, src, dst, t):
        src_mem = self.memory(src)
        dst_mem = self.memory(dst)

        t_enc = self.time_encoding(t)

        # 🔥 time decay
        decay = torch.exp(-t.unsqueeze(1))

        src_mem = src_mem * decay
        dst_mem = dst_mem * decay

        t_enc = self.time_encoding(t)

        score_input = torch.cat([src_mem, dst_mem, t_enc], dim=1)
        score = self.scorer(score_input).squeeze()

        # add time influence
        score = score + t_enc.sum(dim=1)

        score = torch.clamp(score, -5, 5)

        return torch.sigmoid(score)

In [227]:
def prepare(df):
    src = torch.tensor(df['src'].values, dtype=torch.long)
    dst = torch.tensor(df['dst'].values, dtype=torch.long)
    rel = torch.tensor(df['rel'].values, dtype=torch.long)

    time = df['time'].values.astype('datetime64[s]').astype(np.int64)
    time = (time - time.min()) / (time.max() - time.min() + 1e-8)
    time = torch.tensor(time, dtype=torch.float32)

    return src, dst, rel, time

In [228]:
df = df.sort_values('time')

split_time = df['time'].quantile(0.8)

train_df = df[df['time'] < split_time]
test_df = df[df['time'] >= split_time]
train_df_shuffled = train_df.copy()
train_df_shuffled['time'] = np.random.permutation(train_df_shuffled['time'])
train_src, train_dst, train_rel, train_time = prepare(train_df_shuffled)
test_src, test_dst, test_rel, test_time = prepare(test_df)
print(len(train_df), len(test_df))

241 10548


In [229]:
print("Time stats:")
print("min:", train_time.min().item())
print("max:", train_time.max().item())
print("NaNs:", torch.isnan(train_time).sum().item())

Time stats:
min: 0.0
max: 1.0
NaNs: 0


In [230]:
model = RelationalTGN(num_nodes, num_rels, 32)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

for epoch in range(3):
    model.train()
    total_loss = 0

    for i in range(len(train_src)):
        src = train_src[i].unsqueeze(0)
        dst = train_dst[i].unsqueeze(0)
        rel = train_rel[i].unsqueeze(0)
        t = train_time[i].unsqueeze(0).float()

        rel_emb = model.rel_emb(rel)

        pos_pred = model(src, dst, t)

        neg_dst = torch.randint(0, num_nodes, (1,))
        neg_pred = model(src, neg_dst, t)

        loss = loss_fn(pos_pred, torch.ones_like(pos_pred)) + \
               loss_fn(neg_pred, torch.zeros_like(neg_pred))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            src_mem = model.memory(src)
            dst_mem = model.memory(dst)

            msg = model.compute_message(src_mem, dst_mem, rel_emb, t)

            model.update_memory(src, msg)
            model.update_memory(dst, msg)

        total_loss += loss.item()

    print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

Epoch 0, Loss: 329.1471
Epoch 1, Loss: 159.8863
Epoch 2, Loss: 98.9675


In [231]:
model.memory.weight.data.zero_()
with torch.no_grad():
    for i in range(len(train_src)):
        src = train_src[i].unsqueeze(0)
        dst = train_dst[i].unsqueeze(0)
        rel = train_rel[i].unsqueeze(0)
        t = train_time[i].unsqueeze(0).float()

        rel_emb = model.rel_emb(rel)

        src_mem = model.memory(src)
        dst_mem = model.memory(dst)

        msg = model.compute_message(src_mem, dst_mem, rel_emb, t)

        model.update_memory(src, msg)
        model.update_memory(dst, msg)
model.eval()
with torch.no_grad():
    pos_preds = []
    neg_preds = []

    for i in range(len(test_src)):
        src = test_src[i].unsqueeze(0)
        dst = test_dst[i].unsqueeze(0)
        t = test_time[i].unsqueeze(0).float()

        pos_preds.append(model(src, dst, t).item())

        neg_dst = torch.randint(0, num_nodes, (1,))
        neg_preds.append(model(src, neg_dst, t).item())

pos_preds = np.array(pos_preds)
neg_preds = np.array(neg_preds)

y_true = np.concatenate([np.ones(len(pos_preds)), np.zeros(len(neg_preds))])
y_scores = np.concatenate([pos_preds, neg_preds])

auc = roc_auc_score(y_true, y_scores)
print("Test AUC:", auc)

Test AUC: 0.881613230177764


In [232]:
print("Relation embedding norm:", model.rel_emb.weight.norm().item())

Relation embedding norm: 7.326004505157471
